# 29 · CXR-LT — `cxrlt_base.parquet` (Stage 1a)

Dựng bảng khoá cho toàn bộ **227,835 study** của CXR-LT, đúng schema mà tầng đặc trưng EHR cần:

```
study_id  subject_id  index_time  cur_hadm  cur_edstay  setting  split
```

Phần tính `index_time` và join encounter **port nguyên từ `01_cohort_selection.ipynb` C6**, giữ
đúng ngữ nghĩa cũ. Khác một chỗ, và là chỗ có chủ ý:

> nb01 C8 lọc `has_enc = cur_hadm.notna() | cur_edstay.notna()` — cohort cũ **bắt buộc** mọi study
> phải nằm trong một đợt nằm viện hoặc đợt cấp cứu. **Notebook này KHÔNG lọc.** CXR-LT đã định
> nghĩa cohort rồi; ta giữ hết và để `setting='outpatient'` nói lên sự thật thay vì lặng lẽ vứt đi.

Đây cũng là nơi trả lời câu hỏi chặn còn lại của Stage 0: **bao nhiêu study có encounter?** —
cùng một truy vấn, không chạy hai lần.

**CPU + BigQuery, ~10–20 phút.** Quét ~2–4 GB. Cần chạy `28_cxrlt_inventory.ipynb` trước.

## C0 · Config + BigQuery

In [2]:
import os, json, time
import numpy as np, pandas as pd

PROJECT = 'mimic-ehr-v310'
HOSP    = 'physionet-data.mimiciv_3_1_hosp'
CXR     = 'physionet-data.mimic_cxr_jpg'
ED      = None                      # giai o C1 — KHONG doan ten dataset
TMP_DS  = 'cxrlt_tmp'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'

OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
P_IDX   = os.path.join(OUT_DIR, 'cxrlt_index.parquet')
assert os.path.exists(P_IDX), f'khong thay {P_IDX} -> chay 28_cxrlt_inventory.ipynb truoc'

def run_q(sql, label=''):
    '''Dry-run truoc, roi moi chay that.

    Dry-run bat BigQuery phan tich + phan giai ten cho **0 byte bi tinh tien**, nen
    bat duoc `Unrecognized name: x` truoc khi quet mot byte nao, va in ra chi phi
    that. Cac CTE o day du dai de mot loi go phim la chuyen de xay ra.
    '''
    dry = client.query(sql, job_config=bigquery.QueryJobConfig(
        dry_run=True, use_query_cache=False))
    print(f'  [{label}] dry-run OK · se quet {dry.total_bytes_processed/1e9:.2f} GB')
    return client.query(sql).to_dataframe()

IDX = pd.read_parquet(P_IDX, columns=['study_id', 'subject_id', 'split'])
STU = (IDX.drop_duplicates('study_id')
          .sort_values('study_id').reset_index(drop=True))
assert STU.study_id.is_unique
# mot study chi duoc thuoc dung mot split (28 da kiem, kiem lai o day cho chac)
assert IDX.groupby('study_id').split.nunique().max() == 1
print(f'{len(IDX):,} anh -> {len(STU):,} study | split {STU.split.value_counts().to_dict()}')

Mounted at /content/drive
377,110 anh -> 227,835 study | split {'train': 155823, 'test': 48212, 'val': 23800}


## C0-bis · Đăng nhập BigQuery bằng `gcloud`

`auth.authenticate_user()` không dùng được ở đây — tài khoản không phải tài khoản Cloud. Dùng
luồng device-flow: chạy cell dưới, mở link nó in ra, rồi **dán mã trả về ngược lại ô nhập**.

Phải làm lại **mỗi phiên Colab** — ADC không sống qua lần khởi động lại runtime.

In [3]:
!gcloud auth application-default login --no-launch-browser

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=v9IAjCnln5OZWaplzBRnuapwaqkSbv&prompt=consent&token_usage=remote&access_type=offline&code_challenge=gh0ROTEpt3if8Pja9P37V3tiIFZBYU9hWurPaBzNn30&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0AXEQxIAeNQMqBwoRf96Zuc3UB_sawwl7VoYVqDGB-ErYbm00UTWkcYpNJdub340589LFRQ

Credentials saved to file: [/content/.config/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).
Ca

### C0-ter · Gán project rồi thử một query nhỏ

In [4]:
!gcloud config set project {PROJECT}
!gcloud auth application-default set-quota-project {PROJECT} 2>/dev/null || true

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT, location='US')
print('BigQuery OK ->', [dict(r) for r in client.query('SELECT 1 AS ok').result()])


To update your Application Default Credentials quota project, use the `gcloud auth application-default set-quota-project` command.
Updated property [core/project].
BigQuery OK -> [{'ok': 1}]


## C1 · Giải tên dataset ED + kiểm schema `metadata`

Hai thứ **không được đoán**:

1. Tên dataset ED (`mimiciv_ed` / `mimiciv_3_1_ed` / …) đổi theo phiên bản — liệt kê rồi lấy bản
   mới nhất, y như `01_cohort_selection.ipynb` C1.
2. Cột thời gian trong `mimic_cxr_jpg.metadata`. nb01 dùng `study_datetime`; nếu bảng chỉ có
   `StudyDate`/`StudyTime` rời thì ghép tại đây. Kiểm trước để lỗi hiện ra ở dạng đọc được, chứ
   không phải một `Unrecognized name` giữa CTE 40 dòng.

In [5]:
# ---- 1. dataset ED ----
# Phai KHOP PHIEN BAN voi HOSP. Sap xep chuoi roi lay cuoi la SAI: trong
# ['mimiciv_2_2_ed','mimiciv_3_1_ed','mimiciv_ed'] thi sorted()[-1] ra 'mimiciv_ed'
# (vi 'e' > '3'), tuc tron MIMIC-IV v3.1 ben hosp voi mot ban ED khac phien ban.
all_ds = [d.dataset_id for d in client.list_datasets('physionet-data')]
cands  = [d for d in all_ds if d.startswith('mimiciv') and d.split('_')[-1] == 'ed']
assert cands, f'khong thay dataset ED nao trong physionet-data. Co: {sorted(all_ds)[:20]}'

_want = HOSP.split('.')[-1][:-len('_hosp')] + '_ed'      # mimiciv_3_1_hosp -> mimiciv_3_1_ed
if _want in cands:
    ED = f'physionet-data.{_want}'
    print(f'ED  -> {ED}   (khop phien ban voi HOSP)')
else:
    ED = f'physionet-data.{sorted(cands)[-1]}'
    print(f'!! khong co `{_want}` khop phien ban voi HOSP ({HOSP}).')
    print(f'   ung vien: {sorted(cands)}')
    print(f'   -> tam dung {ED}. Kiem xem co lech phien ban khong truoc khi tin ket qua.')

# ---- 2. cot thoi gian cua metadata ----
meta_cols = {f.name for f in client.get_table(f'{CXR}.metadata').schema}
if 'study_datetime' in meta_cols:
    TIME_EXPR = 'CAST(m.study_datetime AS TIMESTAMP)'
    print('metadata: dung cot `study_datetime`')
else:
    assert {'StudyDate', 'StudyTime'} <= meta_cols, \
        (f'metadata khong co `study_datetime` lan cap `StudyDate`/`StudyTime`.\n'
         f'  cac cot co: {sorted(meta_cols)}')
    # StudyDate = 21500712 (INT), StudyTime = 92345.671 (FLOAT, HHMMSS.fff)
    TIME_EXPR = ("PARSE_TIMESTAMP('%Y%m%d%H%M%S', "
                 "CONCAT(CAST(m.StudyDate AS STRING), "
                 "FORMAT('%06d', CAST(FLOOR(m.StudyTime) AS INT64))))")
    print('metadata: ghep `StudyDate` + `StudyTime`')

for _t, _c in [(f'{HOSP}.admissions', {'subject_id','hadm_id','admittime','dischtime'}),
               (f'{ED}.edstays',      {'subject_id','stay_id','intime','outtime'}),
               (f'{HOSP}.patients',   {'subject_id'})]:
    _have = {f.name for f in client.get_table(_t).schema}
    assert _c <= _have, f'{_t} thieu cot {_c - _have}'
print('admissions / edstays / patients: du cot can dung')

!! khong co `mimiciv_3_1_ed` khop phien ban voi HOSP (physionet-data.mimiciv_3_1_hosp).
   ung vien: ['mimiciv_ed']
   -> tam dung physionet-data.mimiciv_ed. Kiem xem co lech phien ban khong truoc khi tin ket qua.
metadata: dung cot `study_datetime`
admissions / edstays / patients: du cot can dung


## C2 · Đẩy danh sách study lên BigQuery

In [6]:
ds = f'{PROJECT}.{TMP_DS}'
client.create_dataset(bigquery.Dataset(ds), exists_ok=True)

keys = STU[['study_id', 'subject_id', 'split']].copy()
keys['study_id']   = keys.study_id.astype('int64')
keys['subject_id'] = keys.subject_id.astype('int64')
client.load_table_from_dataframe(
    keys, f'{ds}.studies',
    job_config=bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')).result()
STU_TBL = f'{ds}.studies'
_n = client.query(f'SELECT COUNT(*) n, COUNT(DISTINCT study_id) d FROM `{STU_TBL}`').to_dataframe()
assert int(_n.n[0]) == len(keys) == int(_n.d[0]), 'so dong tren BigQuery khong khop'
print(f'da day -> {STU_TBL}  ({int(_n.n[0]):,} study)')

da day -> mimic-ehr-v310.cxrlt_tmp.studies  (227,835 study)


## C3 · `index_time` + encounter

Port từ `01_cohort_selection.ipynb` C6. Ngữ nghĩa giữ nguyên từng chữ:

* `index_time` = **thời điểm sớm nhất** của study (`MIN`) — mọi cửa sổ EHR sau này neo vào đây
* `cur_hadm`   = đợt nằm viện **bao trùm** `index_time` (`admittime ≤ index_time ≤ dischtime`)
* `cur_edstay` = đợt cấp cứu **bao trùm** `index_time` (`intime ≤ index_time ≤ outtime`)

Nếu một study rơi vào nhiều đợt (hiếm — thường do bản ghi chồng lấn), `ROW_NUMBER()` lấy đợt **mới
nhất**, và `n_match` ghi lại đã có bao nhiêu đợt khớp để C4 kiểm.

In [7]:
q = f'''
WITH s AS (
  SELECT m.study_id,
         ANY_VALUE(m.subject_id)                    AS subject_id_meta,
         MIN({TIME_EXPR})                           AS index_time,
         COUNT(*)                                   AS n_images_meta,
         COUNT(DISTINCT {TIME_EXPR})                AS n_distinct_times
  FROM `{CXR}.metadata` m
  JOIN `{STU_TBL}` k USING (study_id)
  GROUP BY m.study_id
),
adm AS (
  SELECT study_id, hadm_id, n_match FROM (
    SELECT s.study_id, a.hadm_id,
           COUNT(*)     OVER (PARTITION BY s.study_id)                        AS n_match,
           ROW_NUMBER() OVER (PARTITION BY s.study_id
                              ORDER BY CAST(a.admittime AS TIMESTAMP) DESC)   AS rn
    FROM s
    JOIN `{STU_TBL}` k USING (study_id)
    JOIN `{HOSP}.admissions` a
      ON a.subject_id = k.subject_id
     AND s.index_time BETWEEN CAST(a.admittime AS TIMESTAMP)
                          AND CAST(a.dischtime AS TIMESTAMP)
  ) WHERE rn = 1
),
eds AS (
  SELECT study_id, stay_id, n_match_ed FROM (
    SELECT s.study_id, e.stay_id,
           COUNT(*)     OVER (PARTITION BY s.study_id)                     AS n_match_ed,
           ROW_NUMBER() OVER (PARTITION BY s.study_id
                              ORDER BY CAST(e.intime AS TIMESTAMP) DESC)   AS rn
    FROM s
    JOIN `{STU_TBL}` k USING (study_id)
    JOIN `{ED}.edstays` e
      ON e.subject_id = k.subject_id
     AND s.index_time BETWEEN CAST(e.intime AS TIMESTAMP)
                          AND CAST(e.outtime AS TIMESTAMP)
  ) WHERE rn = 1
),
pat AS (SELECT DISTINCT subject_id, TRUE AS in_patients FROM `{HOSP}.patients`)
SELECT k.study_id, k.subject_id, k.split,
       s.index_time, s.subject_id_meta, s.n_images_meta, s.n_distinct_times,
       adm.hadm_id                  AS cur_hadm,
       COALESCE(adm.n_match, 0)     AS n_hadm_match,
       eds.stay_id                  AS cur_edstay,
       COALESCE(eds.n_match_ed, 0)  AS n_edstay_match,
       COALESCE(pat.in_patients, FALSE) AS in_patients
FROM `{STU_TBL}` k
LEFT JOIN s   USING (study_id)
LEFT JOIN adm USING (study_id)
LEFT JOIN eds USING (study_id)
LEFT JOIN pat ON pat.subject_id = k.subject_id
'''
t0 = time.time()
B = run_q(q, 'base')
print(f've {len(B):,} dong trong {time.time()-t0:.0f}s')
assert len(B) == len(STU), f'{len(B)} != {len(STU)} -> LEFT JOIN nhan doi dong'

  [base] dry-run OK · se quet 0.05 GB
ve 227,835 dong trong 7s


## C4 · `setting` + **độ phủ encounter**

`setting` theo đúng quy tắc `01_cohort_selection.ipynb` C10: có `cur_hadm` → `inpatient`; không
thì có `cur_edstay` → `ed`; còn lại → `outpatient`.

Bảng phủ ở cuối cell này là **đầu ra quan trọng nhất của cả notebook** — nó quyết định cách đọc
mọi con số EHR-only ở Stage 2.

In [8]:
B['setting'] = np.where(B.cur_hadm.notna(), 'inpatient',
                np.where(B.cur_edstay.notna(), 'ed', 'outpatient'))

print('=== DO PHU ENCOUNTER ===')
cov = (B.groupby('setting').size().to_frame('n_study')
         .assign(ty_le=lambda d: (d.n_study / len(B)).map('{:.2%}'.format))
         .reindex(['inpatient', 'ed', 'outpatient']).dropna(how='all'))
print(cov.to_string())
_enc = B.setting.ne('outpatient').mean()
print(f'\nco encounter: {_enc:.2%}   |   ngoai tru: {1-_enc:.2%}')

print('\ntach theo split:')
print(pd.crosstab(B.split, B.setting, normalize='index')
        .reindex(['train', 'val', 'test']).map('{:.2%}'.format).to_string())

print(f'''
Y NGHIA (xem plan, muc Rui ro):
  outpatient MAT VITALS — MIMIC-IV khong co bang vital ngoai tru nao
  (chi co icu.chartevents + ed.vitalsign + ed.triage, deu gan voi encounter).
  Nhung VAN CON: age/gender (100%), comorbidity (tu prior admissions),
  va labs (hosp.labevents co hang hadm_id NULL -> lay duoc qua subject_id).
  -> Stage 2 BAT BUOC bao cao AP tach theo `setting`.''')

# --- canh bao chat luong join ---
print('\n=== KIEM CHAT LUONG ===')
print(f'study khong co index_time      : {int(B.index_time.isna().sum()):,}')
print(f'study khop >1 dot nam vien     : {int((B.n_hadm_match > 1).sum()):,}')
print(f'study khop >1 dot cap cuu      : {int((B.n_edstay_match > 1).sum()):,}')
print(f'study co ca hadm VA edstay     : {int((B.cur_hadm.notna() & B.cur_edstay.notna()).sum()):,}')
print(f'subject KHONG co trong patients: {int((~B.in_patients).sum()):,}'
      f'  -> nhung study nay se khong co age/gender')
_mm = B.subject_id_meta.notna() & (B.subject_id_meta != B.subject_id)
print(f'subject_id lech giua CXR-LT/meta: {int(_mm.sum()):,}')

=== DO PHU ENCOUNTER ===
            n_study   ty_le
setting                    
inpatient    110892  48.67%
ed            77801  34.15%
outpatient    39142  17.18%

co encounter: 82.82%   |   ngoai tru: 17.18%

tach theo split:
setting      ed inpatient outpatient
split                               
train    34.62%    48.22%     17.16%
val      33.11%    49.30%     17.59%
test     33.13%    49.83%     17.04%

Y NGHIA (xem plan, muc Rui ro):
  outpatient MAT VITALS — MIMIC-IV khong co bang vital ngoai tru nao
  (chi co icu.chartevents + ed.vitalsign + ed.triage, deu gan voi encounter).
  Nhung VAN CON: age/gender (100%), comorbidity (tu prior admissions),
  va labs (hosp.labevents co hang hadm_id NULL -> lay duoc qua subject_id).
  -> Stage 2 BAT BUOC bao cao AP tach theo `setting`.

=== KIEM CHAT LUONG ===
study khong co index_time      : 0
study khop >1 dot nam vien     : 0
study khop >1 dot cap cuu      : 1
study co ca hadm VA edstay     : 3,871
subject KHONG co trong patients: 11,

## C5 · Kiểm chứng + ghi `cxrlt_base.parquet`

In [9]:
# --- cong chan: khong co study nao mat index_time ---
_no_t = B.index_time.isna()
if _no_t.any():
    print(f'!! {int(_no_t.sum()):,} study khong co index_time — vi du:')
    print(B.loc[_no_t, ['study_id', 'subject_id', 'split']].head(10).to_string(index=False))
    raise AssertionError(
        f'{int(_no_t.sum())} study khong lay duoc index_time tu {CXR}.metadata.\n'
        '  -> khong the neo cua so EHR cho chung. Kiem xem study_id da cat tien to "s" chua\n'
        '     (28_cxrlt_inventory.ipynb C1 lam viec do), hoac metadata thieu study that.')
assert (B.subject_id_meta.isna() | (B.subject_id_meta == B.subject_id)).all(), \
    'subject_id cua CXR-LT lech voi mimic_cxr_jpg.metadata -> khoa join sai'

BASE = B[['study_id', 'subject_id', 'index_time', 'cur_hadm', 'cur_edstay',
          'setting', 'split', 'n_hadm_match', 'n_edstay_match',
          'n_images_meta', 'in_patients']].copy()
BASE['index_time'] = pd.to_datetime(BASE.index_time)
BASE = BASE.sort_values('study_id').reset_index(drop=True)

assert BASE.study_id.is_unique
assert BASE.index_time.notna().all()
assert set(BASE.setting) <= {'inpatient', 'ed', 'outpatient'}
# split phai khop BIT-FOR-BIT voi 28 — day la thu de vo im lang nhat
_ref = (pd.read_parquet(P_IDX, columns=['study_id', 'split'])
          .drop_duplicates('study_id').sort_values('study_id').reset_index(drop=True))
assert (BASE.study_id.to_numpy() == _ref.study_id.to_numpy()).all()
assert (BASE.split.to_numpy()    == _ref.split.to_numpy()).all(), \
    'split lech so voi cxrlt_index.parquet'
print('kiem chung: study_id duy nhat | index_time day du | split khop 28 bit-for-bit')

P_BASE = os.path.join(OUT_DIR, 'cxrlt_base.parquet')
BASE.to_parquet(P_BASE, index=False)
_c = pd.read_parquet(P_BASE)
assert len(_c) == len(BASE) and _c.index_time.notna().all()
print(f'\nghi {P_BASE}  ({os.path.getsize(P_BASE)/1e6:.1f} MB, {len(_c):,} study)')

print(f'\nindex_time: {BASE.index_time.min()}  ->  {BASE.index_time.max()}')
print(f'''
================================================================
STAGE 1a XONG

  {len(BASE):,} study  |  setting: ''' +
  ' | '.join(f'{k} {v:,}' for k, v in BASE.setting.value_counts().items()) + f'''

  Schema san sang cho tang dac trung EHR:
    study_id  subject_id  index_time  cur_hadm  cur_edstay  setting  split

TIEP THEO -> 29b_cxrlt_ehr_features.ipynb
  Hai bat bien phai giu: (1) join qua subject_id  (2) storetime <= index_time
  cho labs/vitals/comorbidity. Micro co ngoai le [-7d, +72h] tren charttime,
  RIENG nhom ket-qua; nhom chi-bao-da-gui van chan o index_time.
================================================================''')

kiem chung: study_id duy nhat | index_time day du | split khop 28 bit-for-bit

ghi /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/cxrlt_base.parquet  (5.5 MB, 227,835 study)

index_time: 2110-01-11 22:01:11.359000+00:00  ->  2208-12-08 12:04:54.936000+00:00

STAGE 1a XONG

  227,835 study  |  setting: inpatient 110,892 | ed 77,801 | outpatient 39,142

  Schema san sang cho tang dac trung EHR:
    study_id  subject_id  index_time  cur_hadm  cur_edstay  setting  split

TIEP THEO -> 29b_cxrlt_ehr_features.ipynb
  Hai bat bien phai giu: (1) join qua subject_id  (2) storetime <= index_time
  cho labs/vitals/comorbidity. Micro co ngoai le [-7d, +72h] tren charttime,
  RIENG nhom ket-qua; nhom chi-bao-da-gui van chan o index_time.
